## Constructed-data method demonstration: CONSTRUCTED original signal-layer method

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are labeled separately as replays. French raw, extracted, and processed data files are not bundled. French market data are not included or analyzed because rights and permitted uses have not been established.

The code and labels below describe the constructed signal analysis shown here. No French market-data analysis is run in this Notebook.

[Project 4 reader guide](../../../../NOTEBOOK_GUIDE.md)


[Entry modes and preparation](../../../../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

# Project 4 Preparation: Signal Layer (Value / Momentum / Quality) and a Constructed Company Universe

This Notebook calls functions in projects/project04/preparation/signal_layer.py. On a **constructed** company universe, it selects the financial-statement version available by each month-end under point-in-time (PIT) rules; computes the seven raw fields required by the field contract; applies same-date winsorization, industry and size neutralization, z-scoring, and the fixed composite under the frozen rules; and runs six counterexamples (using period_end as available_at; the same-version restatement rule; future-data perturbations; a survivor-only universe; full-sample standardization; and nonpositive denominators). The Notebook uses the current C13 signal algorithm.

- Constructed setup: seed=20261210; 240 companies, including 40 banks; 96 months.
- All data are constructed for teaching (the data-generating process is known, and no value, momentum, or quality effect is built in). The repository contains no point-in-time company filings, prices, or historical equity universe, so the results are not observations about real companies.
- This Notebook covers the signal-preparation layer only. It does not include transaction costs, stock borrowing or financing, capacity, or a walk-forward backtest. Running the first code cell creates a new, unique run directory under runs/p04_signal_layer_<uuid>/; the figure cell writes a PNG to figures/a5/p04_signal_layer.png inside that directory. Run the Notebook from the top, in order.


In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p04_signal_layer_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
spec = importlib.util.spec_from_file_location("p04", ROOT / "projects/project04/preparation/signal_layer.py")
p04 = importlib.util.module_from_spec(spec); spec.loader.exec_module(p04)


## 1. Frozen Rules, Field Contract, and Constructed Universe


In [ ]:
import numpy as np
import pandas as pd

rules, contract = p04.load_rules(), p04.load_contract()
universe = p04.build_universe()
print("frame:", universe["frame"])
print("rules sha256", rules["_sha256"][:16], "| contract sha256", contract["_sha256"][:16])
v = universe["versions"]
print("filings", len(v), "| amendments", int((v["form"] == "10-K/A").sum()), "| names", len(universe["names"]), "| months", len(universe["index"]))
assert universe["frame"]["constructed"] and not universe["frame"]["has_real_company_data"]
pd.DataFrame(contract["signals"]).T[["style", "formula", "direction"]]

## 2. Seven Raw Fields, Coverage, and Same-Date Preprocessing


In [ ]:
raw = p04.compute_signals(universe)
cov = p04.coverage(raw, universe)
z = p04.preprocess(raw, universe, rules)
styles, composite, count = p04.style_scores(z, rules)
assert (composite.notna() == (count >= 2)).to_numpy().all()                          # at least two styles, never a zero-filled composite
checks = pd.DataFrame({k: {"mean |row mean|": v.mean(axis=1).abs().mean(), "mean row sd": v.std(axis=1, ddof=1).mean()} for k, v in z.items()}).T
print(cov.round(3).T.to_string())
checks.round(6)

## 3. Counterexample: Using period_end as available_at (Look-Ahead)


In [ ]:
pe = p04.counterexample_period_end(universe)
table = pd.DataFrame({k: {"share of cells that differ": v["share_of_cells_that_differ"], "leak corr, signal months Feb-Mar": v["leak_channel_corr_feb_mar"]["mean"],
                          "t": v["leak_channel_corr_feb_mar"]["t"], "leak corr, other months": v["leak_channel_corr_other_months"]["mean"]} for k, v in pe.items()}).T
assert table.loc["book_to_market", "t"] > 3 and 0.15 < table.loc["gp_a", "share of cells that differ"] < 0.25
table.round(3)

## 4. Counterexample: The Same-Version Restatement Rule


In [ ]:
rs = p04.counterexample_restatement(universe)
print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in rs.items() if not isinstance(x, dict)})
print(rs["month_before_amendment"], rs["month_of_amendment"])
assert rs["month_before_amendment"]["pit_equity"] == rs["original_equity"] and rs["month_of_amendment"]["pit_equity"] == rs["amended_equity"]

## 5. Counterexample: Future-Data Perturbation (Signals Through the Cutoff Do Not Move)


In [ ]:
fp = p04.counterexample_future_perturbation(universe, rules)
print("cutoff", fp["cutoff"], "| months compared", fp["months_compared"], "| max |difference| up to the cutoff:", max(fp["max_abs_difference_up_to_cutoff"].values()),
      "| composite after the cutoff moves by up to", round(fp["max_abs_difference_of_composite_after_cutoff"], 3))
assert max(fp["max_abs_difference_up_to_cutoff"].values()) == 0.0 and fp["max_abs_difference_of_composite_after_cutoff"] > 0.1

## 6. Counterexamples: Survivor-Only Universe, Nonpositive Denominators, and Full-Sample Standardization


In [ ]:
sv = p04.counterexample_survivorship(universe, rules)
dn = p04.counterexample_denominators(universe)
fs = p04.counterexample_full_sample_standardization(universe, rules)
print("survivors only removes", sv["names_removed_at_date"], "names at", sv["date"], "|", sv["at_date"])
print({k: v for k, v in dn.items()})
print("full-sample vs expanding z-score: opposite sign in", round(100 * fs["share_with_opposite_sign"], 1), "% of the cells")
assert dn["safe_bm_negative_cells"] == 0 < dn["naive_negative_bm_cells"] and 0.05 < fs["share_with_opposite_sign"] < 0.25

## 7. Composite Rules and Exposures (Excluding Costs); IC in a Constructed Zero-Effect World


In [ ]:
d = p04.diagnostics(universe, rules)
print(pd.DataFrame(d["ic"]).T.round(4).to_string())
pd.DataFrame({"plain quintile": d["exposures_quintile_plain"], "industry-neutral projection": d["exposures_quintile_industry_neutral"]}).round(4)

In [ ]:
from IPython.display import Image, display
figure_path = RUN_DIR / p04.FIGURE
p04.make_figure(universe, {"diagnostics":d, "counterexample_period_end":pe}, figure_path)
display(Image(filename=str(figure_path)))


## 8. Try It Yourself

1. Change rules["preprocessing"]["winsorize"]["lower"] to 0.05, then rerun Sections 2 and 7. Write down who should set this choice, when they should set it, and whether it belongs in the ledger.
2. In Section 5, change the cutoff month to "2016-06" and confirm that signals through the cutoff remain unchanged.
3. In Section 3, explain why the leakage correlations for earnings_yield and roe_avg are close to 0, while those for book_to_market and gp_a are not. Hint: in the constructed process, filing news enters assets.
